In [12]:
from utils import simulation
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from concurrent.futures import ProcessPoolExecutor
from tqdm import tqdm
import os
import hashlib
import psutil
import time
import boto3


# AWS S3 Konfiguration
s3 = boto3.client('s3')
bucket_name = 'master-daten'  # Dein S3-Bucket-Name

### Simulations Parameter ##############
seed = 42
n_sim = 10

jk_ab_calc = True
boot_var_calc = False
ijk_std_calc = True
train_models = True

# Funktion zur Überwachung der Ressourcennutzung
def monitor_resources():
    cpu_usage = psutil.cpu_percent(interval=1)
    memory_info = psutil.virtual_memory()
    print(f"CPU usage: {cpu_usage}%")
    print(f"Memory usage: {memory_info.percent}%")
    
def upload_to_s3(local_file_path, s3_file_path):
    try:
        s3.upload_file(local_file_path, bucket_name, s3_file_path)
        print(f"{local_file_path} erfolgreich nach S3 hochgeladen: s3://{bucket_name}/{s3_file_path}")
    except Exception as e:
        print(f"Fehler beim Hochladen von {local_file_path}: {e}")
        
# Funktion, um eine Datei lokal zu löschen
def delete_local_file(local_file_path):
    try:
        os.remove(local_file_path)
        print(f"{local_file_path} erfolgreich lokal gelöscht")
    except Exception as e:
        print(f"Fehler beim Löschen von {local_file_path}: {e}")


for n_i in [100]:
    for _ in [i for i in range(1,2)]:
        ########## Varying Parameters ##########

        n = n_i
        B_RF = int(0.7 * n * _)
        B_1 = 20
        tau = 42
        params_data_creation = { 'shape_weibull': 1, 'scale_weibull_base': 10000, 'rate_censoring': 0.01,
                                'b_bloodp': -0.405, 'b_diab': -0.4, 'b_age': -0.05, 'b_bmi': -0.01, 'b_kreat': -0.2, 
                                'n': n, 'seed': seed, 'tau': tau}

        params_rf = {  'n_estimators':B_RF,                        
                        'max_depth':3,
                        'min_samples_split':5,
                        'max_features': 'log2',
                        'random_state':  seed,
                        'weighted_bootstrapping':     True,  }

        X_erwartung = pd.DataFrame({'bmi': [25], 'blood_pressure': [0], 'kreatinkinase': [np.exp(5+1/2)], 'diabetes': [0], 'age': [50]})

        ######## Start Simulation   ########
        with ProcessPoolExecutor() as executor:
            
            ### Array to store the results
            portion_events_after_cut_train = np.zeros(n_sim)
            portion_censored_after_cut_train = np.zeros(n_sim)
            portion_no_events_after_cut_train = np.zeros(n_sim)
            portion_events_after_cut_test = np.zeros(n_sim)
            portion_censored_after_cut_test = np.zeros(n_sim)
            portion_no_events_after_cut_test = np.zeros(n_sim)
            wb_mse_ipcw = np.zeros(n_sim)
            wb_cindex_ipcw = np.zeros(n_sim)
            wb_y_pred_X_point = np.zeros(n_sim)
            rf_mse_ipcw = np.zeros(n_sim)
            rf_y_pred_X_point = np.zeros(n_sim)
            ijk_var_pred_X_point = np.zeros(n_sim)
            bootstrap_var_pred_X_point = np.zeros(n_sim)
            jk_ab_var_pred_X_point = np.zeros(n_sim)

            futures = [
                executor.submit(
                    simulation,
                    seed=seed+i,
                    tau=tau, 
                    data_generation_weibull_parameters=params_data_creation,
                    params_rf=params_rf, 
                    X_pred_point=X_erwartung,
                    B_first_level = B_1,
                    boot_std_calc = boot_var_calc,
                    ijk_std_calc = ijk_std_calc,
                    train_models = train_models,
                    jk_ab_calc = jk_ab_calc
                )
                for i in range(n_sim)
            ]

            for i, future in enumerate(tqdm(futures, desc="Simulations", unit="simulation")):
                _portion_events_after_cut_train, _portion_censored_after_cut_train, _portion_no_events_after_cut_train, \
                _portion_events_after_cut_test, _portion_censored_after_cut_test, _portion_no_events_after_cut_test, \
                _wb_mse_ipcw, _wb_cindex_ipcw, _wb_y_pred_X_point, \
                _rf_mse_ipcw, _rf_y_pred_X_point, _ijk_var_pred_X_point, _bootstrap_var_pred_X_point, _jk_ab_var_pred_X_point  = future.result()

                if i<3:
                    # Überwache Ressourcen nach jeder Simulation
                    monitor_resources()

                #Event-Stats Results
                portion_events_after_cut_train[i] = _portion_events_after_cut_train
                portion_censored_after_cut_train[i] = _portion_censored_after_cut_train
                portion_no_events_after_cut_train[i] = _portion_no_events_after_cut_train
                portion_events_after_cut_test[i] = _portion_events_after_cut_test
                portion_censored_after_cut_test[i] = _portion_censored_after_cut_test
                portion_no_events_after_cut_test[i] = _portion_no_events_after_cut_test
                
                #Evaluation Results
                wb_mse_ipcw[i] = _wb_mse_ipcw
                wb_cindex_ipcw[i] = _wb_cindex_ipcw
                rf_mse_ipcw[i] = _rf_mse_ipcw

                #Prediction Results
                wb_y_pred_X_point[i] = _wb_y_pred_X_point[0]
                rf_y_pred_X_point[i] = _rf_y_pred_X_point[0]

                # Standard Deviation Estimates
                ijk_var_pred_X_point[i]  = _ijk_var_pred_X_point[0]
                jk_ab_var_pred_X_point[i] = _jk_ab_var_pred_X_point
                bootstrap_var_pred_X_point[i] = _bootstrap_var_pred_X_point

        ########## Save Results ############
        results = pd.DataFrame({'wb_mse_ipcw': wb_mse_ipcw,
                                'wb_cindex_ipcw': wb_cindex_ipcw,
                                'rf_mse_ipcw': rf_mse_ipcw,
                                'wb_y_pred_X_point': wb_y_pred_X_point,
                                'rf_y_pred_X_point': rf_y_pred_X_point,
                                'ijk_var_pred_X_point': ijk_var_pred_X_point,
                                'jk_ab_var_pred_X_point': jk_ab_var_pred_X_point,
                                'bootstrap_var_pred_X_point': bootstrap_var_pred_X_point})
        
        path = os.path.abspath('')
        experiment_name = f'n_train {n*0.7}_Events {round(n*0.7*np.mean(portion_events_after_cut_train),0)} ({round(np.mean(portion_events_after_cut_train)*100,2)} %)_Cens {round(n*0.7*np.mean(portion_censored_after_cut_train),0)} ({round(np.mean(portion_censored_after_cut_train)*100,2)} %)_(B_RF) {B_RF}__(B_1) {B_1}'

        # Erstellen eines lokalen Verzeichnisses, falls es noch nicht existiert
        if not os.path.exists(path + '/results/' + experiment_name):
            os.makedirs(path + '/results/' + experiment_name)

        # Abspeichern der Ergebnisse in einer CSV-Datei
        local_csv_file = path + '/results/' + experiment_name + '/' + experiment_name + '.csv'
        results.to_csv(local_csv_file, index=False)

        # Abspeichern der Ergebnisse in einer TXT-Datei
        local_txt_file = path + '/results/' + experiment_name + '/results.txt'
        with open(local_txt_file, 'w') as f:

            f.write(f'n_train= {n*0.7}// Events: {round(n*0.7*np.mean(portion_events_after_cut_train),0)} ({round(np.mean(portion_events_after_cut_train)*100,2)} %) // Censored: {round(n*0.7*np.mean(portion_censored_after_cut_train),0)} ({round(np.mean(portion_censored_after_cut_train)*100,2)} %) // B_RF: {B_RF} // (B_1): {B_1} \n')

            f.write('\n### Standard Deviation: ###\n')
            a = np.std(wb_y_pred_X_point)
            f.write(f'WB EMP-STD:                 {a:.4f}\n')
            b = np.std(rf_y_pred_X_point)
            f.write(f'RF EMP-STD:                 {b:.4f}\n\n')

            non_neg_var_ijk_est = ijk_var_pred_X_point.copy()
            non_neg_var_ijk_est[non_neg_var_ijk_est<0] = 0
            # erst die wurzel ziehen und dann den mittelwert bilden
            a = np.mean(np.sqrt(  non_neg_var_ijk_est    ))
            f.write(f'IJK STD (for RF) Mean-est               : {a:.4f}  \n rel. Abweichung zu emp. std {(a-b)/b*100:.2f} %  \n std. des schätzers {np.std(np.sqrt(  non_neg_var_ijk_est    )):.2f} \n\n')
            
            non_neg_var_ijk_est = jk_ab_var_pred_X_point.copy()
            non_neg_var_ijk_est[non_neg_var_ijk_est<0] = 0
            # erst die wurzel ziehen und dann den mittelwert bilden
            a = np.mean(np.sqrt(  non_neg_var_ijk_est    ))
            f.write(f'JK-AB(un-weighted) STD (for RF) Mean-est: {a:.4f} \n rel. Abweichung zu emp. std {(a-b)/b*100:.2f} %  \n std. des schätzers {np.std(np.sqrt(  non_neg_var_ijk_est    )):.2f} \n\n')


            # erst die wurzel ziehen und dann den mittelwert bilden
            a = np.mean(np.sqrt(  bootstrap_var_pred_X_point    ))
            f.write(f'Boot STD (for RF) Mean-est              : {a:.4f} \n rel. Abweichung zu emp. std {(a-b)/b*100:.2f} %  \n std. des schätzers {np.std(np.sqrt(  a    )):.2f} \n\n')




            f.write('\n\n### mean Data Stats over all simulations: ###\n')
            f.write('Number of simulations: '+str(n_sim)+'\n')
            f.write('cut-off time tau: '+str(tau)+'\n\n')
            f.write(f'Train ({n*0.7}):\n')
            f.write(f'Events:     {round(np.mean(portion_events_after_cut_train)*100,2)}  %,  n={round(n*0.7*np.mean(portion_events_after_cut_train),0)}\n')
            f.write(f'No Events:  {round(np.mean(portion_no_events_after_cut_train)*100,2)} %,  n={round(n*0.7*np.mean(portion_no_events_after_cut_train),0)}\n')
            f.write(f'Censored:   {round(np.mean(portion_censored_after_cut_train)*100,2)} %,  n={round(n*0.7*np.mean(portion_censored_after_cut_train),0)}\n')
            f.write(f'Test  ({n*0.3}):\n')
            f.write(f'Events:     {round(np.mean(portion_events_after_cut_test)*100,2)}  %,   n={round(n*0.3*np.mean(portion_events_after_cut_test),0)}\n')
            f.write(f'No Events:  {round(np.mean(portion_no_events_after_cut_test)*100,2)} %,   n={round(n*0.3*np.mean(portion_no_events_after_cut_test),0)}\n')
            f.write(f'Censored:   {round(np.mean(portion_censored_after_cut_test)*100,2)}  %,   n={round(n*0.3*np.mean(portion_censored_after_cut_test),0)}\n')


            f.write('\n\n### Evaluation: ###\n')
            f.write(f'WB C-Index IPCW: {np.mean(wb_cindex_ipcw):.4f}\n')
            f.write(f'WB MSE IPCW: {np.mean(wb_mse_ipcw):.4f}\n')
            f.write(f'RF MSE IPCW: {np.mean(rf_mse_ipcw):.4f}\n')
            f.write('\n')

            f.write('\n###Prediction Results:###\n')
            f.write(f'True Y: {99999999999999:.4f}\n')
            f.write(f'WB Y_pred: {np.mean(wb_y_pred_X_point):.4f}\n')
            f.write(f'RF Y_pred: {np.mean(rf_y_pred_X_point):.4f}\n')
            f.write('\n')



            f.write('\n\n### Simulation Parameters: ###\n')
            f.write('first_level_B for bootstrap variance estimation (B_1): '+str(B_1)+'\n')

            f.write('Data Creation Parameters:\n')
            f.write(str(params_data_creation))
            
            f.write('\nRandom Forest Parameter:\n')
            f.write(str(params_rf)+'\n')
            f.write(f'the above seeds ({seed}) are start_seed for the simulation function\n')
        
        ########## Upload zu S3 ############
        s3_csv_path = 'results/' + experiment_name + '/' + experiment_name + '.csv'
        s3_txt_path = 'results/' + experiment_name + '/results.txt'
        
        # Lade CSV und TXT zu S3 hoch
        upload_to_s3(local_csv_file, s3_csv_path)
        upload_to_s3(local_txt_file, s3_txt_path)

        ########## Lokale Dateien löschen ############
        delete_local_file(local_csv_file)
        delete_local_file(local_txt_file)
        
        
        # Generiere den Hash für den Experimentnamen
        hash_object = hashlib.md5(experiment_name.encode())
        experiment_hash = hash_object.hexdigest()

        # Definiere den Pfad für das Verzeichnis und die Datei
        local_txt_dir = path + '/results_txt/' + str(experiment_hash)
        local_txt_file = local_txt_dir + '/results.txt'

        # Erstelle das Verzeichnis, falls es nicht existiert
        if not os.path.exists(local_txt_dir):
            os.makedirs(local_txt_dir)

        with open(local_txt_file, 'w') as f:

            f.write(f'n_train= {n*0.7}// Events: {round(n*0.7*np.mean(portion_events_after_cut_train),0)} ({round(np.mean(portion_events_after_cut_train)*100,2)} %) // Censored: {round(n*0.7*np.mean(portion_censored_after_cut_train),0)} ({round(np.mean(portion_censored_after_cut_train)*100,2)} %) // B_RF: {B_RF} // (B_1): {B_1} \n')


            f.write('\n### Standard Deviation: ###\n')
            a = np.std(wb_y_pred_X_point)
            f.write(f'WB EMP-STD:                 {a:.4f}\n')
            b = np.std(rf_y_pred_X_point)
            f.write(f'RF EMP-STD:                 {b:.4f}\n\n')

            non_neg_var_ijk_est = ijk_var_pred_X_point.copy()
            non_neg_var_ijk_est[non_neg_var_ijk_est<0] = 0
            # erst die wurzel ziehen und dann den mittelwert bilden
            a = np.mean(np.sqrt(  non_neg_var_ijk_est    ))
            f.write(f'IJK STD (for RF) Mean-est               : {a:.4f}  \n rel. Abweichung zu emp. std {(a-b)/b*100:.2f} %  \n std. des schätzers {np.std(np.sqrt(  non_neg_var_ijk_est    )):.2f} \n\n')
            
            non_neg_var_ijk_est = jk_ab_var_pred_X_point.copy()
            non_neg_var_ijk_est[non_neg_var_ijk_est<0] = 0
            # erst die wurzel ziehen und dann den mittelwert bilden
            a = np.mean(np.sqrt(  non_neg_var_ijk_est    ))
            f.write(f'JK-AB(un-weighted) STD (for RF) Mean-est: {a:.4f} \n rel. Abweichung zu emp. std {(a-b)/b*100:.2f} %  \n std. des schätzers {np.std(np.sqrt(  non_neg_var_ijk_est    )):.2f} \n\n')

            # erst die wurzel ziehen und dann den mittelwert bilden
            a = np.mean(np.sqrt(  bootstrap_var_pred_X_point    ))
            f.write(f'Boot STD (for RF) Mean-est              : {a:.4f} \n rel. Abweichung zu emp. std {(a-b)/b*100:.2f} %  \n std. des schätzers {np.std(np.sqrt(  a    )):.2f} \n\n')

            f.write('\n\n### mean Data Stats over all simulations: ###\n')
            f.write('Number of simulations: '+str(n_sim)+'\n')
            f.write('cut-off time tau: '+str(tau)+'\n\n')
            f.write(f'Train ({n*0.7}):\n')
            f.write(f'Events:     {round(np.mean(portion_events_after_cut_train)*100,2)}  %,  n={round(n*0.7*np.mean(portion_events_after_cut_train),0)}\n')
            f.write(f'No Events:  {round(np.mean(portion_no_events_after_cut_train)*100,2)} %,  n={round(n*0.7*np.mean(portion_no_events_after_cut_train),0)}\n')
            f.write(f'Censored:   {round(np.mean(portion_censored_after_cut_train)*100,2)} %,  n={round(n*0.7*np.mean(portion_censored_after_cut_train),0)}\n')
            f.write(f'Test  ({n*0.3}):\n')
            f.write(f'Events:     {round(np.mean(portion_events_after_cut_test)*100,2)}  %,   n={round(n*0.3*np.mean(portion_events_after_cut_test),0)}\n')
            f.write(f'No Events:  {round(np.mean(portion_no_events_after_cut_test)*100,2)} %,   n={round(n*0.3*np.mean(portion_no_events_after_cut_test),0)}\n')
            f.write(f'Censored:   {round(np.mean(portion_censored_after_cut_test)*100,2)}  %,   n={round(n*0.3*np.mean(portion_censored_after_cut_test),0)}\n')


            f.write('\n\n### Evaluation: ###\n')
            f.write(f'WB C-Index IPCW: {np.mean(wb_cindex_ipcw):.4f}\n')
            f.write(f'WB MSE IPCW: {np.mean(wb_mse_ipcw):.4f}\n')
            f.write(f'RF MSE IPCW: {np.mean(rf_mse_ipcw):.4f}\n')
            f.write('\n')

            f.write('\n###Prediction Results:###\n')
            f.write(f'True Y: {99999999999999:.4f}\n')
            f.write(f'WB Y_pred: {np.mean(wb_y_pred_X_point):.4f}\n')
            f.write(f'RF Y_pred: {np.mean(rf_y_pred_X_point):.4f}\n')
            f.write('\n')



            f.write('\n\n### Simulation Parameters: ###\n')
            f.write('first_level_B for bootstrap variance estimation (B_1): '+str(B_1)+'\n')

            f.write('Data Creation Parameters:\n')
            f.write(str(params_data_creation))
            
            f.write('\nRandom Forest Parameter:\n')
            f.write(str(params_rf)+'\n')
            f.write(f'the above seeds ({seed}) are start_seed for the simulation function\n')
        
        # Definiere den S3-Pfad für die Datei
        s3_txt_path = 'results_txt/' + str(experiment_hash) + '/results.txt'

        # Lade die Datei nach S3 hoch
        upload_to_s3(local_txt_file, s3_txt_path)
        
        delete_local_file(local_txt_file)
        print(experiment_name+'   finished')


Simulations:  10%|█         | 1/10 [00:02<00:18,  2.07s/simulation]

CPU usage: 100.0%
Memory usage: 32.6%


Simulations:  20%|██        | 2/10 [00:03<00:11,  1.44s/simulation]

CPU usage: 100.0%
Memory usage: 32.6%


Simulations: 100%|██████████| 10/10 [00:04<00:00,  2.46simulation/s]

CPU usage: 90.0%
Memory usage: 32.6%
/root/results/n_train 70.0_Events 13.0 (18.86 %)_Cens 22.0 (31.43 %)_(B_RF) 70__(B_1) 20/n_train 70.0_Events 13.0 (18.86 %)_Cens 22.0 (31.43 %)_(B_RF) 70__(B_1) 20.csv erfolgreich nach S3 hochgeladen: s3://master-daten/results/n_train 70.0_Events 13.0 (18.86 %)_Cens 22.0 (31.43 %)_(B_RF) 70__(B_1) 20/n_train 70.0_Events 13.0 (18.86 %)_Cens 22.0 (31.43 %)_(B_RF) 70__(B_1) 20.csv
/root/results/n_train 70.0_Events 13.0 (18.86 %)_Cens 22.0 (31.43 %)_(B_RF) 70__(B_1) 20/results.txt erfolgreich nach S3 hochgeladen: s3://master-daten/results/n_train 70.0_Events 13.0 (18.86 %)_Cens 22.0 (31.43 %)_(B_RF) 70__(B_1) 20/results.txt
/root/results/n_train 70.0_Events 13.0 (18.86 %)_Cens 22.0 (31.43 %)_(B_RF) 70__(B_1) 20/n_train 70.0_Events 13.0 (18.86 %)_Cens 22.0 (31.43 %)_(B_RF) 70__(B_1) 20.csv erfolgreich lokal gelöscht
/root/results/n_train 70.0_Events 13.0 (18.86 %)_Cens 22.0 (31.43 %)_(B_RF) 70__(B_1) 20/results.txt erfolgreich lokal gelöscht


/root/results_txt/ab1ae83b10d59e3560d65bb036a5dc5f/results.txt erfolgreich nach S3 hochgeladen: s3://master-daten/results_txt/ab1ae83b10d59e3560d65bb036a5dc5f/results.txt
/root/results_txt/ab1ae83b10d59e3560d65bb036a5dc5f/results.txt erfolgreich lokal gelöscht
n_train 70.0_Events 13.0 (18.86 %)_Cens 22.0 (31.43 %)_(B_RF) 70__(B_1) 20   finished


In [11]:
#!pip install lifelines scikit-survival seaborn

/root/results_txt/ab1ae83b10d59e3560d65bb036a5dc5f/results.txt erfolgreich nach S3 hochgeladen: s3://master-daten/results_txt/ab1ae83b10d59e3560d65bb036a5dc5f/results.txt
